# Optimizer Landscapes and Trajectories

Visualize how different optimizers traverse loss surfaces on controlled objectives.

In [ ]:
import sys

sys.path.insert(0, "src")

import numpy as np
import matplotlib.pyplot as plt

from from_zero.optim import (
    SGD,
    Momentum,
    RMSProp,
    Adam,
    make_objective,
    run_optimizer,
    OptimizerComparisonConfig,
    compare_optimizers,
)

## 1. Quadratic Objective (Convex, Ill-Conditioned)

f(x, y) = 0.5 * (c1 * x^2 + c2 * y^2) with c1=1, c2=10

In [ ]:
obj_quad = make_objective("quadratic", curvature=(1.0, 10.0))

optimizers = {
    "SGD": SGD(0.1),
    "Momentum": Momentum(0.1, beta=0.9),
    "RMSProp": RMSProp(0.1, rho=0.99),
    "Adam": Adam(0.1),
}

results = {}
for name, opt in optimizers.items():
    res = run_optimizer(obj_quad, opt, steps=100)
    results[name] = res
    print(f"{name}: final_loss={res.final_loss:.2e}, steps={res.steps}")

In [ ]:
# Plot trajectories on contour
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)
Z = 0.5 * (1.0 * X**2 + 10.0 * Y**2)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (name, res) in zip(axes, results.items()):
    traj = res.trajectory
    ax.contour(X, Y, Z, levels=np.logspace(-2, 2, 20), alpha=0.5)
    ax.plot(traj[:, 0], traj[:, 1], "o-", markersize=2, label=name)
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 2. Rosenbrock Function (Narrow Valley)

f(x, y) = (1-x)^2 + 100(y-x^2)^2 — global minimum at (1, 1)

In [ ]:
obj_rosen = make_objective("rosenbrock")

results_rosen = {}
for name, opt in optimizers.items():
    opt.reset_state()
    res = run_optimizer(obj_rosen, opt, steps=500)
    results_rosen[name] = res
    print(f"{name}: final_loss={res.final_loss:.4f}, steps={res.steps}")

In [ ]:
# Rosenbrock contour
x = np.linspace(-1.5, 2, 100)
y = np.linspace(-0.5, 3, 100)
X, Y = np.meshgrid(x, y)
Z = (1 - X) ** 2 + 100 * (Y - X**2) ** 2

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (name, res) in zip(axes, results_rosen.items()):
    traj = res.trajectory
    ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), alpha=0.5)
    ax.plot(traj[:, 0], traj[:, 1], "o-", markersize=2, label=name)
    ax.plot(1, 1, "r*", markersize=15, label="Optimum")
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 3. Logistic Regression Objective

Real ML loss landscape on a small 2D dataset.

In [ ]:
obj_log = make_objective("logistic", n_samples=200, noise=0.15, seed=1)

results_log = {}
for name, opt in optimizers.items():
    opt.reset_state()
    res = run_optimizer(obj_log, opt, steps=200)
    results_log[name] = res
    print(f"{name}: final_loss={res.final_loss:.4f}, steps={res.steps}")

In [ ]:
# Loss curves comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res_dict), title in zip(
    axes, [results, results_rosen, results_log], ["Quadratic", "Rosenbrock", "Logistic"]
):
    for name, res in res_dict.items():
        ax.semilogy(res.losses, label=name)
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss (log)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Full Comparison (Documented Run)

Run the full comparison with objective-specific learning rates and 1000 steps.

In [ ]:
config = OptimizerComparisonConfig(
    name="notebook_comparison",
    seed=123,
    steps=1000,
    optimizers=("sgd", "momentum", "rmsprop", "adam"),
    objectives=("quadratic", "rosenbrock", "logistic"),
    output_dir="artifacts/experiments",
    make_plots=True,
)

comparison = compare_optimizers(config)

print("Rankings:")
for obj_name, runs in comparison.runs.items():
    sorted_runs = sorted(runs.items(), key=lambda x: x[1].final_loss)
    ranking = " > ".join(f"{name}({run.final_loss:.2e})" for name, run in sorted_runs)
    print(f"  {obj_name}: {ranking}")